# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [5]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [6]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [7]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [8]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [9]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [10]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [11]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [12]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [13]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [14]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [15]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [16]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.
This will create a manager automatically.


In [17]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [18]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [19]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
I need to delegate tasks to the right team members in order to effectively monitor and analyze market data for AAPL.

Action: Delegate work to co-worker
Action Input: {
    "coworker": "Data Analyst",
    "task": "Continuously monitor and analyze market data for AAPL using statistical modeling and machine learning",
    "context": "The task involves analyzing historical and real-time market data for AAPL, identifying trends, and predicting market movements using statistical modeling and machine learning techniques."
}

> Entering new CrewAgentExecutor chain...
I need to gather as much information as possible about AAPL's market data to provide accurate insights to my coworker.

Action: Read website content
Action 

Thought: I need to gather insights and predictions from the market data analysis for AAPL in order to recommend the best trading strategies.

Action: Search the internet
Action Input: {"search_query": "market data analysis for AAPL trading strategies"} 


Search results: Title: Technical Analysis of Apple Inc (NASDAQ:AAPL) - TradingView
Link: https://www.tradingview.com/symbols/NASDAQ-AAPL/technicals/
Snippet: This gauge displays a technical analysis overview for your selected timeframe. The summary of Apple Inc is based on the most popular technical indicators, ...
---
Title: APPLE INC(NASDAQ:AAPL) stock Technical Analysis | ChartMill.com
Link: https://www.chartmill.com/stock/quote/AAPL/technical-analysis
Snippet: When comparing the yearly performance of all stocks, we notice that AAPL is one of the better performing stocks in the market, outperforming 79% of all stocks.
---
Title: Apple Trading Strategies & Insights - LinkedIn
Link: https://www.linkedin.com/pulse/apple-trading-strate

Final Answer: After analyzing the approved trading strategies for AAPL from various sources such as TradingView, ChartMill.com, LinkedIn, Vestinda, and Nasdaq, the best execution methods will involve utilizing technical analysis tools, exploring different trading approaches, analyzing chart patterns, and considering factor-based stock analysis. It is crucial to stay informed about mixed analyst opinions and stock forecasts to make well-informed trading decisions.

> Finished chain.
 

After analyzing the approved trading strategies for AAPL from various sources such as TradingView, ChartMill.com, LinkedIn, Vestinda, and Nasdaq, the best execution methods will involve utilizing technical analysis tools, exploring different trading approaches, analyzing chart patterns, and considering factor-based stock analysis. It is crucial to stay informed about mixed analyst opinions and stock forecasts to make well-informed trading decisions.


Thought: 
I need to ask the Trading Strategy Developer

I need to gather as much information as possible from TradingView and ChartMill.com to evaluate the proposed trading strategies and execution plans for AAPL.

Action: Search the internet
Action Input: {"search_query": "AAPL technical analysis tools TradingView ChartMill.com"} 


Search results: Title: APPLE INC(NASDAQ:AAPL) stock Technical Analysis | ChartMill.com
Link: https://www.chartmill.com/stock/quote/AAPL/technical-analysis
Snippet: We assign a technical rating of 5 out of 10 to AAPL. AAPL was one of the better performers in the overall market, it is still ok in the medium term time ...
---
Title: Technical Analysis of Apple Inc (NASDAQ:AAPL) - TradingView
Link: https://www.tradingview.com/symbols/NASDAQ-AAPL/technicals/
Snippet: Which technical analysis tools can be used to analyze Apple Inc? Check out various oscillators, moving averages and other technical indicators on ...
---
Title: NASDAQ:AAPL Stock Price — TradingView - Apple Inc
Link: https://www.tradingview.com/symbols/

- Display the final result as Markdown.

In [20]:
from IPython.display import Markdown
Markdown(result)

Risk Analysis Report for AAPL Trading Strategies:

After evaluating the proposed trading strategies and execution plans for AAPL, the following risks have been identified along with mitigation recommendations:

1. Market Volatility Risk:
- With the current market conditions and uncertainties, there is a risk of increased market volatility affecting AAPL's stock price.
Mitigation: Implement stop-loss orders to limit potential losses and diversify the portfolio to reduce overall market risk exposure.

2. Technical Analysis Limitations:
- Relying solely on technical analysis tools may not always accurately predict market trends and price movements.
Mitigation: Combine technical analysis with fundamental analysis to gain a more comprehensive view of AAPL's performance.

3. Analyst Recommendation Risk:
- Analyst opinions and stock forecasts may vary, leading to conflicting recommendations.
Mitigation: Consider a consensus approach by analyzing a wide range of analyst opinions and trends to make informed decisions.

4. Support and Resistance Levels Risk:
- Fluctuations in support and resistance levels can impact trading decisions and lead to potential losses.
Mitigation: Utilize dynamic stop-loss orders based on updated support and resistance levels to adapt to changing market conditions.

5. Market News and Events Risk:
- Unexpected market news or events can significantly impact AAPL's stock price and market performance.
Mitigation: Stay informed about market news and events, and consider setting up alerts to react promptly to any significant developments.

By addressing these risks and implementing the suggested mitigation strategies, the team can better navigate the challenges in trading AAPL and make well-informed decisions to optimize trading outcomes.